In [39]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from statsmodels.stats.inter_rater import fleiss_kappa

# Import CSV file
df = pd.read_csv('QwenMicrosoftQwenSmallNormal.csv')

# Rows that contain "error" as output
error_mask = df['Output'] == 'error'


num_errors = error_mask.sum()

# Replace the 'error' strings with unique values (error1, error2, ...)
# Done to ensure that "errors" aren't rewarded as agreeing outputs
df.loc[error_mask, 'Output'] = [f'error{i+1}' for i in range(num_errors)]

print(f"Replaced {num_errors} instances of 'error' with unique identifiers.")

# Count num of sat/unsat/(unknown)/(error) per document-scenario pair
kappa_matrix = pd.crosstab(df['RuleScenario'], df['Output'])

# Verify that every document-scenario pair has 3 outputs associated (one for each model)
# NOTE: MUST BE CHANGED BASED ON NUMBER OF TRANSLATION MODELS USED
# Also drops rows with duplicates, which happens on very small cases like the Johnny toy example
row_sums = kappa_matrix.sum(axis=1)
if not (row_sums == 3).all():
    print("Warning: Some Rule Texts do not have exactly 3 model evaluations.")
    print("Check these Rule Texts for missing or duplicate data:\n",
          row_sums[row_sums != 3])

    print("\nDropping rows where the sum of evaluations is not exactly 3...\n")
    kappa_matrix = kappa_matrix[row_sums == 3]

# 3. Calculate Fleiss' Kappa
kappa_score = fleiss_kappa(kappa_matrix.values)

print(f"Fleiss' Kappa Score: {kappa_score:.4f}")

# 4. Calculate P-Value & Statistical Significance

N = kappa_matrix.shape[0] # Number of doc-scenario pairs
n = 3 # Number of raters (models)

# Proportions of all assignments to each category
p_j = kappa_matrix.sum(axis=0) / (N * n)

# Calculate components for the variance formula
sum_pj2 = np.sum(p_j**2)
sum_pj3 = np.sum(p_j**3)

# Variance of Kappa under the null hypothesis (Kappa = 0)
var_kappa = (2 / (N * n * (n - 1))) * (sum_pj2 - (2*n - 3) * (sum_pj2**2) + 2 * (n - 2) * sum_pj3) / ((1 - sum_pj2)**2)
se_kappa = np.sqrt(var_kappa)

# Calculate Z-score and two-tailed p-value
z_score = kappa_score / se_kappa
p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

print(f"Standard Error: {se_kappa:.4f}")
print(f"Z-score: {z_score:.4f}")
print(f"P-value: {p_value:.4e}")


Replaced 2 instances of 'error' with unique identifiers.
Check these Rule Texts for missing or duplicate data:
 RuleScenario
Rule Text: To be eligible for the standard residential lease, the applicant must be at least 18 years of age and possess a credit score of 650 or higher. If the applicant is under 18 years of age, they are strictly prohibited from leasing, regardless of credit score. If the applicant is 18 or older but has a credit score below 650, they may only be approved if they provide a qualified guarantor.\n\n\nScenario: 18, 550, Guarantor Present    6
dtype: int64

Dropping rows where the sum of evaluations is not exactly 3...

Fleiss' Kappa Score: 0.6815
Standard Error: 0.1063
Z-score: 6.4092
P-value: 1.4628e-10

The Fleiss' Kappa score is statistically significant (p < 0.05).


In [21]:
# Drop all duplicates based on 'RuleScenario' and 'ModelName'.
# keep=False ensures that ALL instances of the duplicates are removed, not just the extra ones.
comp_df = df.drop_duplicates(subset=['RuleScenario', 'ModelName'], keep=False)

print(f"Original dataframe shape: {df.shape}")
print(f"Deduplicated dataframe shape: {comp_df.shape}")


Original dataframe shape: (90, 5)
Deduplicated dataframe shape: (78, 5)


,ModelName,ModelID,RuleScenario,SMTCode,Output
1,Dolphin-2.9.1-Yi-1.5-34B,dphn/dolphin-2.9.1-yi-1.5-34b,"Rule Text: If Johnny is running late, he will ...",(declare-const johnny_running_late Bool)\n(dec...,sat
2,Dolphin-2.9.1-Yi-1.5-34B,dphn/dolphin-2.9.1-yi-1.5-34b,"Rule Text: If Johnny is running late, he will ...",(declare-const johnny_running_late Bool)\n(dec...,sat
4,Dolphin-2.9.1-Yi-1.5-34B,dphn/dolphin-2.9.1-yi-1.5-34b,"Rule Text: If Johnny is running late, he will ...",(declare-const johnny_running_late Bool)\n(dec...,sat
5,Dolphin-2.9.1-Yi-1.5-34B,dphn/dolphin-2.9.1-yi-1.5-34b,"Rule Text: If Johnny is running late, he will ...",(declare-const johnny_running_late Bool)\n(dec...,sat
6,Dolphin-2.9.1-Yi-1.5-34B,dphn/dolphin-2.9.1-yi-1.5-34b,"Rule Text: If Johnny is running late, he will ...",(declare-const johnny_running_late Bool)\n(dec...,sat


In [1]:
"""
SORTS dataframe to help find disagreements and compare scripts closely.
"""
comp_df = comp_df.sort_values(by="RuleScenario")

# CHANGE to your preferred file name
comp_df.to_csv("./YOURNEWNAME.csv", index=False)

NameError: name 'comp_df' is not defined

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.inter_rater import fleiss_kappa

def bootstrap_fleiss_ci(matrix, n_iterations=1000, ci=95):
    """Calculates a bootstrapped Confidence Interval for Fleiss' Kappa."""
    n_subjects = matrix.shape[0]
    bootstrapped_kappas = []

    for _ in range(n_iterations):
        # Randomly resample the rows (Doc-Scenario pairs) with replacement
        indices = np.random.choice(n_subjects, size=n_subjects, replace=True)
        resampled_matrix = matrix[indices]

        # Calculate Kappa for this random sample
        k = fleiss_kappa(resampled_matrix)
        bootstrapped_kappas.append(k)

    # Calculate the lower and upper bounds
    alpha = (100 - ci) / 2
    lower_bound = np.percentile(bootstrapped_kappas, alpha)
    upper_bound = np.percentile(bootstrapped_kappas, 100 - alpha)

    return lower_bound, upper_bound

# Run the bootstrap
lower, upper = bootstrap_fleiss_ci(kappa_matrix.values)

print(f"Fleiss' Kappa: {kappa_score:.4f}")
print(f"95% Confidence Interval: [{lower:.4f}, {upper:.4f}]")

In [3]:
"""
MOUNT to Google Drive if necessary when using Google Colab.
"""

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

os.chdir('/your/directory/here') # Changes the directory
print(os.getcwd())

/content/drive/MyDrive/Colab Notebooks/CPSC 4900
